# 02 MMTIS Loading and Preparation

In this notebook, we load the actual train operation data from the MMTIS dataset.

The goal is to calculate train delays and prepare clean datasets for later visualizations.

## 1. Import libraries

First, we import the libraries that we need for data loading and basic data preparation.

In [51]:
import pandas as pd
from pathlib import Path

## 2. Define data paths

We define the path to the raw MMTIS data and to the folder where we will save processed files.

In [52]:
raw_path = Path("../data/raw/mmtis")
processed_path = Path("../data/processed")

processed_path.mkdir(parents=True, exist_ok=True)

## 3. Find all train operation files

The MMTIS folder contains weekly CSV files.  
We use the files named `zugfahrten.csv`, because they contain all train trips, not only delayed trains.

In [53]:
trip_files = sorted(
    raw_path.rglob("*_zugfahrten.csv")
)

len(trip_files)

27

In [54]:
trip_files[:5]

[PosixPath('../data/raw/mmtis/2025/2025-12-01_mmtis_zugfahrten.csv'),
 PosixPath('../data/raw/mmtis/2025/2025-12-08_mmtis_zugfahrten.csv'),
 PosixPath('../data/raw/mmtis/2025/2025-12-15_mmtis_zugfahrten.csv'),
 PosixPath('../data/raw/mmtis/2025/2025-12-22_mmtis_zugfahrten.csv'),
 PosixPath('../data/raw/mmtis/2025/2025-12-29_mmtis_zugfahrten.csv')]

## 4. Load all train operation files

We load all weekly train operation files and combine them into one dataset.

In [55]:
dfs = []

for file in trip_files:
    df = pd.read_csv(file)
    dfs.append(df)

trips = pd.concat(dfs, ignore_index=True)

In [56]:
trips.shape

(1053324, 9)

In [57]:
trips.head()

,zugnummer,betriebstag,abfahrtzeit_soll,abfahrtzeit_ist,start_betriebsstelle,ankunftzeit_soll,ankunftzeit_ist,ziel_betriebsstelle,fahrzeit_delta
0,20,2025-11-17,2025-11-17T17:15:00+0100,2025-11-17T17:19:13+0100,MLX,NaN,2025-11-17T19:53:37+0100,PAG,NaN
1,21,2025-11-17,2025-11-17T10:21:30+0100,2025-11-17T10:34:31+0100,PAG,2025-11-17T12:44:45+0100,2025-11-17T12:49:46+0100,MLX,-00:08:00
2,22,2025-11-17,2025-11-17T15:15:00+0100,2025-11-17T15:15:42+0100,MLX,NaN,2025-11-17T17:41:51+0100,PAG,NaN
3,23,2025-11-17,2025-11-17T12:22:54+0100,2025-11-17T12:56:25+0100,PAG,2025-11-17T14:44:45+0100,2025-11-17T15:18:47+0100,MLX,00:00:31
4,26,2025-11-17,2025-11-17T11:15:00+0100,2025-11-17T11:19:33+0100,MLX,NaN,2025-11-17T13:37:02+0100,PAG,NaN


## 5. Inspect the dataset

Now we check the columns, data types, and missing values.

In [58]:
trips.columns.tolist()

['zugnummer',
 'betriebstag',
 'abfahrtzeit_soll',
 'abfahrtzeit_ist',
 'start_betriebsstelle',
 'ankunftzeit_soll',
 'ankunftzeit_ist',
 'ziel_betriebsstelle',
 'fahrzeit_delta']

In [59]:
trips.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1053324 entries, 0 to 1053323
Data columns (total 9 columns):
 #   Column                Non-Null Count    Dtype 
---  ------                --------------    ----- 
 0   zugnummer             1053324 non-null  int64 
 1   betriebstag           1053324 non-null  object
 2   abfahrtzeit_soll      1049472 non-null  object
 3   abfahrtzeit_ist       1047032 non-null  object
 4   start_betriebsstelle  1053324 non-null  object
 5   ankunftzeit_soll      1044231 non-null  object
 6   ankunftzeit_ist       1046983 non-null  object
 7   ziel_betriebsstelle   1053324 non-null  object
 8   fahrzeit_delta        1034256 non-null  object
dtypes: int64(1), object(8)
memory usage: 72.3+ MB


In [60]:
trips.isna().sum()

zugnummer                   0
betriebstag                 0
abfahrtzeit_soll         3852
abfahrtzeit_ist          6292
start_betriebsstelle        0
ankunftzeit_soll         9093
ankunftzeit_ist          6341
ziel_betriebsstelle         0
fahrzeit_delta          19068
dtype: int64

## 6. Convert time columns

The time columns are loaded as text.  
We convert them to datetime format so that we can calculate delays.

In [61]:
time_columns = [
    "abfahrtzeit_soll",
    "abfahrtzeit_ist",
    "ankunftzeit_soll",
    "ankunftzeit_ist"
]

for col in time_columns:
    trips[col] = pd.to_datetime(
        trips[col],
        utc=True,
        errors="coerce"
    )

In [62]:
trips[time_columns].dtypes

abfahrtzeit_soll    datetime64[ns, UTC]
abfahrtzeit_ist     datetime64[ns, UTC]
ankunftzeit_soll    datetime64[ns, UTC]
ankunftzeit_ist     datetime64[ns, UTC]
dtype: object

In [63]:
trips["abfahrtzeit_soll"].head()

0   2025-11-17 16:15:00+00:00
1   2025-11-17 09:21:30+00:00
2   2025-11-17 14:15:00+00:00
3   2025-11-17 11:22:54+00:00
4   2025-11-17 10:15:00+00:00
Name: abfahrtzeit_soll, dtype: datetime64[ns, UTC]

## 7. Calculate delays

We calculate departure delay and arrival delay in minutes.

A positive value means that the train was late.  
A negative value means that the train was earlier than scheduled.

In [64]:
trips["departure_delay_min"] = (
    trips["abfahrtzeit_ist"] - trips["abfahrtzeit_soll"]
).dt.total_seconds() / 60

In [65]:
trips["arrival_delay_min"] = (
    trips["ankunftzeit_ist"] - trips["ankunftzeit_soll"]
).dt.total_seconds() / 60

In [66]:
trips[[
    "zugnummer",
    "betriebstag",
    "departure_delay_min",
    "arrival_delay_min"
]].head()

,zugnummer,betriebstag,departure_delay_min,arrival_delay_min
0,20,2025-11-17,4.216667,NaN
1,21,2025-11-17,13.016667,5.016667
2,22,2025-11-17,0.700000,NaN
3,23,2025-11-17,33.516667,34.033333
4,26,2025-11-17,4.550000,NaN


## 8. Create time features

We create simple time features from the scheduled departure time.

These features will help us analyze delays by hour, weekday, month, and time of day.

In [67]:
trips["departure_hour"] = trips["abfahrtzeit_soll"].dt.hour
trips["weekday"] = trips["abfahrtzeit_soll"].dt.day_name()
trips["month"] = trips["abfahrtzeit_soll"].dt.month_name()

In [68]:
trips[[
    "abfahrtzeit_soll",
    "departure_hour",
    "weekday",
    "month"
]].head()

,abfahrtzeit_soll,departure_hour,weekday,month
0,2025-11-17 16:15:00+00:00,16.0,Monday,November
1,2025-11-17 09:21:30+00:00,9.0,Monday,November
2,2025-11-17 14:15:00+00:00,14.0,Monday,November
3,2025-11-17 11:22:54+00:00,11.0,Monday,November
4,2025-11-17 10:15:00+00:00,10.0,Monday,November


## 9. Create time-of-day categories

We group the departure hour into simple categories: morning peak, midday, evening peak, and night.

In [69]:
def get_time_of_day(hour):
    if pd.isna(hour):
        return "unknown"
    elif 6 <= hour < 9:
        return "morning_peak"
    elif 9 <= hour < 16:
        return "midday"
    elif 16 <= hour < 19:
        return "evening_peak"
    else:
        return "night"

In [70]:
trips["time_of_day"] = trips["departure_hour"].apply(get_time_of_day)

In [71]:
trips["time_of_day"].value_counts()

time_of_day
midday          382504
night           341442
evening_peak    166303
morning_peak    159223
unknown           3852
Name: count, dtype: int64

## 10. Prepare delay analysis dataset

Some records do not have scheduled or actual arrival times.  
For delay analysis, we keep only records where arrival delay can be calculated.

In [72]:
delay_analysis = trips.dropna(
    subset=["arrival_delay_min"]
).copy()

In [73]:
delay_analysis.shape

(1037890, 15)

In [74]:
delay_analysis["arrival_delay_min"].describe()

count    1.037890e+06
mean     1.925526e+00
std      7.032093e+00
min     -5.604833e+02
25%     -1.833333e-01
50%      4.333333e-01
75%      2.016667e+00
max      1.137217e+03
Name: arrival_delay_min, dtype: float64

## 11. Handle extreme delay values

Some delay values are very large or very negative.  
For visualizations, we create a capped delay column.

This keeps the main patterns visible and reduces the effect of extreme outliers.

In [75]:
delay_analysis["arrival_delay_capped"] = (
    delay_analysis["arrival_delay_min"]
    .clip(lower=-30, upper=120)
)

In [76]:
delay_analysis[[
    "arrival_delay_min",
    "arrival_delay_capped"
]].describe()

,arrival_delay_min,arrival_delay_capped
count,1.037890e+06,1.037890e+06
mean,1.925526e+00,1.921824e+00
std,7.032093e+00,6.172415e+00
min,-5.604833e+02,-3.000000e+01
25%,-1.833333e-01,-1.833333e-01
50%,4.333333e-01,4.333333e-01
75%,2.016667e+00,2.016667e+00
max,1.137217e+03,1.200000e+02


## 12. Check punctuality

We define a train as on time if it arrives no more than 5 minutes late.

In [77]:
delay_analysis["on_time"] = (
    delay_analysis["arrival_delay_min"] <= 5
)

In [78]:
on_time_rate = delay_analysis["on_time"].mean()

on_time_rate

np.float64(0.902030080258987)

## 13. Create summary table by hour

This table shows the average arrival delay for each departure hour.

In [79]:
hour_summary = (
    delay_analysis
    .groupby("departure_hour", as_index=False)
    .agg(
        mean_delay=("arrival_delay_capped", "mean"),
        median_delay=("arrival_delay_capped", "median"),
        number_of_trips=("arrival_delay_capped", "count")
    )
)

hour_summary

,departure_hour,mean_delay,median_delay,number_of_trips
0,0.0,1.063688,-0.066667,3021
1,1.0,1.394905,0.083333,2764
2,2.0,1.482556,0.216667,12529
3,3.0,1.338393,0.300000,40434
4,4.0,1.723193,0.433333,62136
5,5.0,2.034995,0.566667,64892
6,6.0,2.188402,0.533333,55720
7,7.0,2.060592,0.416667,51014
8,8.0,1.820895,0.366667,49680
9,9.0,1.776980,0.366667,48432


## 14. Create summary table by weekday

This table shows the average arrival delay for each weekday.

In [80]:
weekday_summary = (
    delay_analysis
    .groupby("weekday", as_index=False)
    .agg(
        mean_delay=("arrival_delay_capped", "mean"),
        median_delay=("arrival_delay_capped", "median"),
        number_of_trips=("arrival_delay_capped", "count")
    )
)

weekday_summary

,weekday,mean_delay,median_delay,number_of_trips
0,Friday,2.208855,0.550000,155824
1,Monday,2.011663,0.516667,153853
2,Saturday,1.345812,0.200000,132538
3,Sunday,1.326890,0.150000,127091
4,Thursday,2.118489,0.566667,153397
5,Tuesday,2.166582,0.566667,155158
6,Wednesday,2.074099,0.550000,156416


## 15. Create summary table by station pair

This table shows average delay between start and destination operating points.

In [81]:
station_pair_summary = (
    delay_analysis
    .groupby(
        ["start_betriebsstelle", "ziel_betriebsstelle"],
        as_index=False
    )
    .agg(
        mean_delay=("arrival_delay_capped", "mean"),
        median_delay=("arrival_delay_capped", "median"),
        number_of_trips=("arrival_delay_capped", "count")
    )
)

In [82]:
station_pair_summary.sort_values(
    "mean_delay",
    ascending=False
).head(20)

,start_betriebsstelle,ziel_betriebsstelle,mean_delay,median_delay,number_of_trips
3256,PG,PG G,120.0,120.0,1
1624,JB,LN,120.0,120.0,1
690,ESU,ZE,120.0,120.0,1
3615,SBM,MAG,120.0,120.0,1
1897,KTF,MZ,120.0,120.0,1
2189,LIR,BG,120.0,120.0,1
357,BOB,GZS,120.0,120.0,1
3631,SBM,VB,120.0,120.0,1
3654,SCHG,MLX,120.0,120.0,1
3214,PAG,PW,120.0,120.0,2


## 16. Save processed datasets

We save the prepared trip-level data and the summary tables.  
These files will be used later for visualizations and the dashboard.

In [83]:
delay_analysis.to_csv(
    processed_path / "mmtis_delay_analysis.csv",
    index=False
)

hour_summary.to_csv(
    processed_path / "mmtis_hour_summary.csv",
    index=False
)

weekday_summary.to_csv(
    processed_path / "mmtis_weekday_summary.csv",
    index=False
)

station_pair_summary.to_csv(
    processed_path / "mmtis_station_pair_summary.csv",
    index=False
)

In [84]:
print("Saved files:")
print(processed_path / "mmtis_delay_analysis.csv")
print(processed_path / "mmtis_hour_summary.csv")
print(processed_path / "mmtis_weekday_summary.csv")
print(processed_path / "mmtis_station_pair_summary.csv")

Saved files:
../data/processed/mmtis_delay_analysis.csv
../data/processed/mmtis_hour_summary.csv
../data/processed/mmtis_weekday_summary.csv
../data/processed/mmtis_station_pair_summary.csv


## 17. Basic data quality checks

Before moving to visualizations, we check whether the processed delay dataset looks reasonable.

We check:
- number of rows
- missing values
- delay distribution
- very early and very late trains
- number of unique station pairs

In [85]:
print("Rows in delay_analysis:", delay_analysis.shape[0])
print("Columns in delay_analysis:", delay_analysis.shape[1])

delay_analysis[
    [
        "arrival_delay_min",
        "arrival_delay_capped",
        "departure_delay_min",
        "departure_hour",
        "weekday",
        "month",
        "time_of_day"
    ]
].describe()

Rows in delay_analysis: 1037890
Columns in delay_analysis: 17


,arrival_delay_min,arrival_delay_capped,departure_delay_min,departure_hour
count,1.037890e+06,1.037890e+06,1.034256e+06,1.034277e+06
mean,1.925526e+00,1.921824e+00,1.346237e+00,1.184714e+01
std,7.032093e+00,6.172415e+00,2.555378e+01,5.653794e+00
min,-5.604833e+02,-3.000000e+01,-1.584000e+04,0.000000e+00
25%,-1.833333e-01,-1.833333e-01,3.333333e-02,7.000000e+00
50%,4.333333e-01,4.333333e-01,5.000000e-01,1.200000e+01
75%,2.016667e+00,2.016667e+00,1.316667e+00,1.600000e+01
max,1.137217e+03,1.200000e+02,1.131783e+03,2.300000e+01


## 18. Missing values in the final dataset

We check missing values after filtering the data for valid arrival timestamps.
This helps us understand which columns are safe to use for analysis and visualization.

In [86]:
delay_analysis.isna().sum().sort_values(ascending=False).head(20)

fahrzeit_delta          3634
departure_delay_min     3634
abfahrtzeit_soll        3613
month                   3613
weekday                 3613
departure_hour          3613
abfahrtzeit_ist           21
arrival_delay_min          0
arrival_delay_capped       0
time_of_day                0
zugnummer                  0
betriebstag                0
ziel_betriebsstelle        0
ankunftzeit_ist            0
ankunftzeit_soll           0
start_betriebsstelle       0
on_time                    0
dtype: int64

## 19. Check extreme delay values

Some delay values are very large or very negative.  
These observations may be caused by cancellations, data errors, or special operational cases.

For visualizations, we use a capped delay metric, but we still inspect the original values.

In [87]:
delay_analysis.sort_values("arrival_delay_min").head(10)[
    [
        "zugnummer",
        "start_betriebsstelle",
        "ziel_betriebsstelle",
        "ankunftzeit_soll",
        "ankunftzeit_ist",
        "arrival_delay_min"
    ]
]

,zugnummer,start_betriebsstelle,ziel_betriebsstelle,ankunftzeit_soll,ankunftzeit_ist,arrival_delay_min
259102,2155,SH,SH,2026-01-03 04:07:00+00:00,2026-01-02 18:46:31+00:00,-560.483333
155167,3440,AU,AU,2025-12-15 03:45:00+00:00,2025-12-14 21:07:06+00:00,-397.900000
372503,5015,W,W,2026-01-23 06:17:00+00:00,2026-01-23 00:15:53+00:00,-361.116667
58022,3902,SL,SL,2025-11-27 05:11:00+00:00,2025-11-26 23:32:49+00:00,-338.183333
174291,5625,BG H1,BG H1,2025-12-18 05:09:00+00:00,2025-12-17 23:33:09+00:00,-335.850000
286593,3780,KG,KG,2026-01-08 04:27:00+00:00,2026-01-07 22:51:42+00:00,-335.300000
166226,2107,GM,GM,2025-12-17 04:09:00+00:00,2025-12-16 23:07:08+00:00,-301.866667
922183,21094,TU,TU,2026-04-30 02:59:00+00:00,2026-04-29 21:58:22+00:00,-300.633333
494256,21094,TU,TU,2026-02-13 03:59:00+00:00,2026-02-12 23:00:26+00:00,-298.566667
46220,3400,AT,AT,2025-11-25 03:38:00+00:00,2025-11-24 22:46:00+00:00,-292.000000


In [88]:
delay_analysis.sort_values("arrival_delay_min", ascending=False).head(10)[
    [
        "zugnummer",
        "start_betriebsstelle",
        "ziel_betriebsstelle",
        "ankunftzeit_soll",
        "ankunftzeit_ist",
        "arrival_delay_min"
    ]
]

,zugnummer,start_betriebsstelle,ziel_betriebsstelle,ankunftzeit_soll,ankunftzeit_ist,arrival_delay_min
997807,13154,JSW,VBO,2026-05-14 06:17:00+00:00,2026-05-15 01:14:13+00:00,1137.216667
561542,25862,GOTA1,WON,2026-02-25 16:25:49+00:00,2026-02-26 05:28:33+00:00,782.733333
301853,294,TBV,SBM,2026-01-11 05:55:47+00:00,2026-01-11 13:24:52+00:00,449.083333
749320,1036,MLX,MLX,2026-03-31 16:41:49+00:00,2026-04-01 00:09:05+00:00,447.266667
739613,4424,LZW,STG,2026-03-29 20:17:30+00:00,2026-03-30 03:18:30+00:00,421.000000
525252,4630,GU Z2,LIEH1,2026-02-19 20:14:01+00:00,2026-02-20 02:33:55+00:00,379.900000
301827,234,VBO,MLX,2026-01-11 08:03:54+00:00,2026-01-11 14:21:52+00:00,377.966667
933552,161,MT,MLX,2026-05-03 12:29:34+00:00,2026-05-03 17:57:01+00:00,327.450000
933996,1034,HE,BELG,2026-05-03 13:32:29+00:00,2026-05-03 18:50:27+00:00,317.966667
312977,1031,BPA,OF,2026-01-13 09:19:25+00:00,2026-01-13 14:25:19+00:00,305.900000


## 20. Create heatmap summary table

This table shows average delay by weekday and hour.  
It will be used later for the time heatmap in the dashboard.

In [89]:
heatmap_summary = (
    delay_analysis
    .groupby(["weekday", "departure_hour"])
    .agg(
        mean_delay=("arrival_delay_capped", "mean"),
        median_delay=("arrival_delay_min", "median"),
        number_of_trips=("arrival_delay_min", "count")
    )
    .reset_index()
)

heatmap_summary.head()

,weekday,departure_hour,mean_delay,median_delay,number_of_trips
0,Friday,0.0,1.144629,-0.233333,211
1,Friday,1.0,1.570074,0.033333,225
2,Friday,2.0,1.418238,0.233333,1899
3,Friday,3.0,1.463994,0.333333,6492
4,Friday,4.0,1.930933,0.500000,9796


In [90]:
heatmap_summary.to_csv(
    processed_path / "mmtis_heatmap_summary.csv",
    index=False
)

## 21. Create top delayed station pairs table

For the dashboard, we prepare a ranked table of station pairs with the highest average delay.

To make the result more reliable, we only keep station pairs with at least 30 observations.

In [91]:
top_station_pairs = (
    station_pair_summary
    .query("number_of_trips >= 30")
    .sort_values("mean_delay", ascending=False)
    .head(30)
)

top_station_pairs

,start_betriebsstelle,ziel_betriebsstelle,mean_delay,median_delay,number_of_trips
3201,PA,MLX,38.968315,26.083333,91
2547,MLX,ND G,33.543229,14.991667,32
2475,MKI,I W1,23.970000,24.733333,45
383,BPA,VBO,22.715568,15.433333,455
3210,PAG,MLX,21.901109,11.666667,1307
1346,HE,BC,20.516781,4.983333,146
2525,MLX,JB,19.913864,14.900000,113
1896,KTF,MLX,19.604688,4.608333,32
4722,ZD,BVSH1,19.545556,17.416667,30
1548,HW Z3,NIFG,18.929688,1.250000,32


In [92]:
top_station_pairs.to_csv(
    processed_path / "mmtis_top_station_pairs.csv",
    index=False
)